In [ ]:
import os

In [6]:
os.chdir('..')

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [8]:
from kidney_cancer.constants import  *
from kidney_cancer.utils.common import read_yaml, create_directories

In [9]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [10]:
import os
import zipfile
import gdown
from kidney_cancer import logger
from kidney_cancer.utils.common import get_size

In [11]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        """
        Fetch data from the url
        """

        try:
            dataset_url = self.config.source_url
            zip_download_dir = self.config.local_data_file
            os.makedirs(self.config.root_dir, exist_ok=True)

            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?export=download&id="
            gdown.download(prefix+file_id, zip_download_dir)

            logger.info("Downloading dataset from %s into file %s",dataset_url,zip_download_dir)
        
        except Exception as e:
            raise e
    
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-07-26 12:02:20,513: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-26 12:02:20,520: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-26 12:02:20,522: INFO: common: created directory at: artifacts]
[2025-07-26 12:02:20,523: INFO: common: created directory at: artifacts/data_ingestion]


Downloading...
From (original): https://drive.google.com/uc?export=download&id=1aMYYXIjTlXadTDs1TlDYh2pMY0xLmeTd
From (redirected): https://drive.google.com/uc?export=download&id=1aMYYXIjTlXadTDs1TlDYh2pMY0xLmeTd&confirm=t&uuid=1c3c0124-b653-4017-8ecc-19851b3220c3
To: c:\Users\User\Desktop\DS\Kidney_cancer\artifacts\data_ingestion\data.zip
100%|██████████| 940M/940M [00:59<00:00, 15.8MB/s] 

[2025-07-26 12:03:22,417: INFO: 3405117146: Downloading dataset from https://drive.google.com/file/d/1aMYYXIjTlXadTDs1TlDYh2pMY0xLmeTd/view?usp=sharing into file artifacts/data_ingestion/data.zip]
